# 07 — Lựa chọn mô hình và báo cáo cuối
Notebook này tổng hợp kết quả đã tạo bởi pipeline chuẩn. Xếp hạng chỉ dựa trên Validation; Test chỉ dùng báo cáo sau khi quyết định đã khóa.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
reports=ROOT/'reports'
summary=json.loads((reports/'summary.json').read_text(encoding='utf-8'))

## 1. Kiểm tra giao thức thực nghiệm

In [ ]:
display(pd.DataFrame({'rows':summary['splitRows'],'ratio':summary['splitRatios']}))
train_dates=set(summary['splitGroups']['train']); validation_dates=set(summary['splitGroups']['validation']); test_dates=set(summary['splitGroups']['test'])
assert train_dates.isdisjoint(validation_dates) and train_dates.isdisjoint(test_dates) and validation_dates.isdisjoint(test_dates)
print('Không có ngày thu thập giao nhau giữa ba tập.')

## 2. Bảng so sánh trên Validation
F2 là tiêu chí chốt vì với SOC, bỏ sót tấn công thường nguy hiểm hơn một cảnh báo cần kiểm tra thêm.

In [ ]:
comparison=pd.read_csv(reports/'model_comparison.csv').sort_values(['f2','recall','precision'],ascending=False)
display(comparison[['model','best_params','cv_best_f2','accuracy','precision','recall','f1','f2','roc_auc','average_precision']])

## 3. So sánh trực quan các metric
Không kết luận từ Accuracy một mình. Precision phản ánh tải cảnh báo giả; Recall phản ánh tỷ lệ tấn công được phát hiện.

In [ ]:
metrics=['accuracy','precision','recall','f1','f2','roc_auc','average_precision']
ax=comparison.set_index('model')[metrics].plot.bar(figsize=(14,6),ylim=(0,1.05))
ax.set_ylabel('Validation score'); ax.set_title('So sánh mô hình — VALIDATION'); plt.xticks(rotation=25,ha='right'); plt.tight_layout(); plt.show()

## 4. Quy tắc lựa chọn đã khóa
Mô hình có Validation F2 cao nhất được chọn. Ensemble không mặc định là người thắng; Voting chỉ được chọn nếu điểm Validation thực sự tốt hơn mô hình đơn.

In [ ]:
selected_validation=comparison.iloc[0]
print('Theo bảng Validation:',selected_validation['model'])
print('Artifact chính thức:',summary['selectedModel'])
assert selected_validation['model']==summary['selectedModel']
display(selected_validation.to_frame('selected'))

## 5. Mở Test đúng một lần
Không dùng bảng này để quay lại đổi feature, siêu tham số hoặc mô hình. Nếu thay thiết kế, cần một Test mới độc lập.

In [ ]:
test_metrics=pd.Series(summary['testMetrics'],name='Untouched Test')
display(test_metrics.to_frame())
display(Image(filename=str(reports/'confusion_matrix.png')))
display(Image(filename=str(reports/'roc_curve.png')))

## 6. Ý nghĩa SOC của Confusion Matrix
- **TP:** log tấn công được phát hiện.
- **FN:** tấn công bị bỏ sót — rủi ro cần ưu tiên giảm.
- **FP:** log bình thường bị cảnh báo — làm tăng tải analyst.
- **TN:** log bình thường được bỏ qua đúng.

Dataset không cung cấp chi phí tiền tệ thực tế, vì vậy không tự suy diễn ROI từ các số đếm.

In [ ]:
# Đọc số đếm từ dự đoán không được lưu riêng; hiển thị ảnh chính thức và diễn giải từ metric.
recall=test_metrics['recall']; precision=test_metrics['precision']
print(f'Mô hình phát hiện khoảng {recall:.2%} mẫu tấn công được gán nhãn trong Test.')
print(f'Trong các cảnh báo dương của mô hình, khoảng {precision:.2%} khớp nhãn tấn công của dataset.')
print('Đây là agreement với nhãn rule-derived, không phải tỷ lệ hiệu quả production đã được chứng minh.')

## 7. Kiểm tra khoảng cách Validation–Test
Khoảng cách lớn có thể báo hiệu overfit theo giai đoạn hoặc concept drift.

In [ ]:
selected_metrics=selected_validation[metrics].astype(float)
common=[m for m in metrics if m in test_metrics.index]
gap=pd.DataFrame({'validation':selected_metrics[common],'test':test_metrics[common]})
gap['absolute_gap']=(gap.validation-gap.test).abs()
display(gap)
gap[['validation','test']].plot.bar(figsize=(11,5),ylim=(0,1.05),title='Validation và Test'); plt.tight_layout(); plt.show()

## 8. Kết luận phương pháp
- **Train:** học preprocessing, tham số và chạy K-Fold.
- **Validation:** chọn mô hình/ngưỡng; chưa phải kết luận cuối.
- **Test:** chỉ báo cáo sau khi khóa quyết định.
- **Ensemble:** chỉ dùng khi có bằng chứng Validation tốt hơn.
- **Không refit tùy tiện:** refit Train+Validation thay đổi mô hình đã đánh giá; nếu refit để triển khai cần hiệu chuẩn/ngưỡng mới bằng out-of-fold hoặc một validation mới.
- **Khả năng triển khai:** chạy shadow mode, lưu model version/evidence và theo dõi drift trước khi tự động hóa incident.

## 9. Giới hạn bắt buộc khi bảo vệ
- Nhãn bắt nguồn từ MITRE/Wazuh rule, không phải ground truth độc lập.
- Dữ liệu là môi trường Windows APT mô phỏng.
- Không chứng minh hiệu năng cho VPN/SSO, web, Linux, brute-force hoặc zero-day.
- Điểm classifier không phải xác suất tấn công đã hiệu chỉnh.
- Cần external validation bằng log Wazuh thật của SecuraAI trước production.

## 10. Kết luận cuối
Mục tiêu của phần ML là chứng minh nhóm hiểu quy trình xây dựng, đánh giá và tích hợp mô hình vào hệ thống phần mềm. Kết quả không nhằm tuyên bố vượt qua Wazuh, SOC analyst hoặc LLM lớn.